# 🎬 Colab Video Generator - AI-Powered Content to Video

**Tự động chuyển đổi bài viết thành video chuyên nghiệp với AI, TTS, Lip Sync**

---

## ⚙️ BƯỚC 1: CÀI ĐẶT VÀ KHỞI TẠO

Chạy ô này trước tiên để cài đặt tất cả dependencies

In [ ]:
# Cài đặt dependencies chính
!pip install -q torch torchvision torchaudio 2>/dev/null
!pip install -q trafilatura requests beautifulsoup4 2>/dev/null
!pip install -q google-generativeai 2>/dev/null
!pip install -q edge-tts 2>/dev/null
!pip install -q moviepy imageio imageio-ffmpeg 2>/dev/null
!pip install -q opencv-python 2>/dev/null
!pip install -q gradio 2>/dev/null
!pip install -q librosa soundfile 2>/dev/null
!pip install -q numpy scipy scikit-image 2>/dev/null
!pip install -q langdetect pycountry 2>/dev/null
!pip install -q pillow tqdm 2>/dev/null
!pip install -q transformers diffusers safetensors 2>/dev/null

print("✅ Tất cả dependencies đã cài đặt!")

## 🔑 BƯỚC 2: THIẾT LẬP API KEY VÀ GOOGLE DRIVE

Cấp quyền truy cập API Gemini và Google Drive

In [ ]:
import os
import json
from google.colab import userdata

# Lấy Gemini API Key từ Colab Secrets
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print("✅ Gemini API Key đã được cấu hình")
except Exception as e:
    print(f"⚠️ Lỗi: {e}")
    print("📝 Hướng dẫn: Vào 'Secrets' (khóa) -> Thêm secret 'GEMINI_API_KEY' từ https://makersuite.google.com/app/apikeys")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive đã được kết nối")

# Tạo thư mục output
output_dir = '/content/drive/MyDrive/colab-video-generator'
os.makedirs(f'{output_dir}/videos', exist_ok=True)
os.makedirs(f'{output_dir}/checkpoints', exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)
print(f"✅ Thư mục output đã được tạo: {output_dir}")

## 📦 BƯỚC 3: IMPORT THƯ VIỆN VÀ ĐỊNH NGHĨA HÀM CHÍNH

Khai báo tất cả module cần thiết

In [ ]:
import torch
import cv2
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple
import asyncio
import edge_tts
from moviepy.editor import *
import trafilatura
import google.generativeai as genai
from langdetect import detect
from tqdm import tqdm
import logging

# Cấu hình logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Tất cả thư viện đã được import")

## 🛠️ BƯỚC 4: ĐỊNH NGHĨA CÁC HÀM XỬ LÝ

### 4.1: Web Scraping - Lấy nội dung từ URL

In [ ]:
def scrape_content(url: str) -> Dict[str, str]:
    """
    Lấy nội dung từ URL
    """
    try:
        logger.info(f"🔍 Đang tải nội dung từ: {url}")
        
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            logger.error(f"❌ Không thể tải: {url}")
            return None
        
        content = trafilatura.extract(downloaded)
        title = trafilatura.extract_meta(downloaded, 'title')
        
        logger.info(f"✅ Đã lấy {len(content)} ký tự từ {url}")
        
        return {
            'url': url,
            'title': title,
            'content': content,
            'timestamp': datetime.now().isoformat()
        }
    except Exception as e:
        logger.error(f"❌ Lỗi khi scrape: {e}")
        return None

print("✅ Hàm scrape_content đã được định nghĩa")

### 4.2: Gemini API - Tóm tắt và chia cảnh

In [ ]:
def summarize_with_gemini(content: str, language: str = 'vi') -> Dict:
    """
    Dùng Gemini API để tóm tắt nội dung và chia thành các cảnh
    """
    try:
        genai.configure(api_key=os.environ.get('GEMINI_API_KEY'))
        model = genai.GenerativeModel('gemini-pro')
        
        # Prompt theo ngôn ngữ
        prompts = {
            'vi': f"""Hãy tóm tắt nội dung sau và chia thành 3-5 cảnh chi tiết cho video. 
            Mỗi cảnh cần có:
            - Tiêu đề cảnh
            - Lời thoại/kịch bản (script)
            - Mô tả hình ảnh (visual prompt)
            - Thời lượng (giây)
            Trả về dạng JSON với cấu trúc: {"summary": "...", "scenes": [{"title": "...", "script": "...", "visual_prompt": "...", "duration": 10}]}
            
            Nội dung:\n{content[:2000]}""",
            'en': f"""Summarize the content and divide into 3-5 video scenes.
            Each scene needs:
            - Scene title
            - Script/narration
            - Visual description (visual prompt)
            - Duration (seconds)
            Return JSON format: {{"summary": "...", "scenes": [{{"title": "...", "script": "...", "visual_prompt": "...", "duration": 10}}]}}
            
            Content:\n{content[:2000]}"""
        }
        
        prompt = prompts.get(language, prompts['en'])
        logger.info(f"📝 Đang tóm tắt với Gemini ({language})...")
        
        response = model.generate_content(prompt)
        result_text = response.text
        
        # Thử parse JSON
        try:
            json_start = result_text.find('{')
            json_end = result_text.rfind('}') + 1
            if json_start != -1 and json_end > json_start:
                result = json.loads(result_text[json_start:json_end])
            else:
                result = {'summary': result_text, 'scenes': []}
        except:
            result = {'summary': result_text, 'scenes': []}
        
        logger.info(f"✅ Đã tóm tắt thành {len(result.get('scenes', []))} cảnh")
        return result
    
    except Exception as e:
        logger.error(f"❌ Lỗi Gemini: {e}")
        return {'summary': content[:500], 'scenes': []}

print("✅ Hàm summarize_with_gemini đã được định nghĩa")

### 4.3: Text-to-Speech (Edge-TTS) - Tạo voice-over

In [ ]:
async def generate_voiceover(text: str, language: str = 'vi', voice: str = None, output_path: str = None) -> str:
    """
    Tạo file âm thanh từ text bằng Edge-TTS
    """
    try:
        if not voice:
            # Giọng mặc định theo ngôn ngữ
            voices_map = {
                'vi': 'vi-VN-NhanNeural',
                'en': 'en-US-AriaNeural',
                'zh': 'zh-CN-XiaoxuanNeural',
                'ja': 'ja-JP-NanamiNeural',
                'ko': 'ko-KR-SunHiNeural',
                'es': 'es-ES-ElviraNeural',
                'fr': 'fr-FR-DeniseNeural',
                'de': 'de-DE-AmalaNeural',
            }
            voice = voices_map.get(language, 'en-US-AriaNeural')
        
        if not output_path:
            output_path = f'/tmp/voiceover_{datetime.now().timestamp()}.mp3'
        
        logger.info(f"🎤 Tạo voice-over với {voice}...")
        
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(output_path)
        
        logger.info(f"✅ Voice-over tạo xong: {output_path}")
        return output_path
    
    except Exception as e:
        logger.error(f"❌ Lỗi TTS: {e}")
        return None

def generate_voiceover_sync(text: str, language: str = 'vi', voice: str = None, output_path: str = None) -> str:
    """
    Wrapper để gọi async function từ notebook
    """
    return asyncio.run(generate_voiceover(text, language, voice, output_path))

print("✅ Hàm generate_voiceover đã được định nghĩa")

### 4.4: Tạo Video Placeholder - Thay thế cho Video Generation (tiết kiệm VRAM)

In [ ]:
def create_video_from_images(image_paths: List[str], audio_path: str, fps: int = 24, output_path: str = None) -> str:
    """
    Tạo video từ danh sách ảnh và kết hợp với âm thanh
    Đây là phương pháp tiết kiệm VRAM thay vì dùng Stable Video Diffusion
    """
    try:
        if not output_path:
            output_path = f'/tmp/video_{datetime.now().timestamp()}.mp4'
        
        logger.info(f"🎥 Tạo video từ {len(image_paths)} ảnh...")
        
        # Load âm thanh
        if audio_path and os.path.exists(audio_path):
            audio = AudioFileClip(audio_path)
            duration = audio.duration
            logger.info(f"📊 Âm thanh: {duration:.2f}s")
        else:
            duration = len(image_paths) / fps
            audio = None
        
        # Tạo video clips từ ảnh
        clips = []
        img_duration = duration / len(image_paths) if image_paths else 1
        
        for img_path in image_paths:
            if os.path.exists(img_path):
                clip = ImageClip(img_path).set_duration(img_duration)
                clips.append(clip)
        
        if not clips:
            logger.error("❌ Không tìm thấy ảnh nào")
            return None
        
        # Ghép các clips
        video = concatenate_videoclips(clips)
        
        # Thêm âm thanh
        if audio:
            video = video.set_audio(audio)
        
        # Xuất video
        video.write_videofile(output_path, fps=fps, verbose=False, logger=None)
        
        logger.info(f"✅ Video tạo xong: {output_path}")
        
        # Giải phóng bộ nhớ
        video.close()
        if audio:
            audio.close()
        
        return output_path
    
    except Exception as e:
        logger.error(f"❌ Lỗi tạo video: {e}")
        return None

print("✅ Hàm create_video_from_images đã được định nghĩa")

### 4.5: Tạo placeholder ảnh (thay thế Video Generation)

In [ ]:
def create_placeholder_images(visual_prompts: List[str], num_images: int = 3, output_dir: str = None) -> List[str]:
    """
    Tạo ảnh placeholder (gradient với text) thay vì dùng diffusion model
    Để tiết kiệm VRAM và tăng tốc độ
    """
    try:
        if not output_dir:
            output_dir = '/tmp/video_frames'
        os.makedirs(output_dir, exist_ok=True)
        
        image_paths = []
        
        for idx, prompt in enumerate(visual_prompts[:num_images]):
            # Tạo ảnh gradient
            width, height = 1280, 720
            img = np.zeros((height, width, 3), dtype=np.uint8)
            
            # Gradient từ xanh đến tím
            for i in range(height):
                color_ratio = i / height
                img[i, :] = [
                    int(255 * (1 - color_ratio)),  # Blue
                    int(100 * color_ratio),         # Green
                    int(255 * color_ratio)          # Red
                ]
            
            # Thêm text
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.8
            color = (255, 255, 255)
            thickness = 2
            
            # Wrap text nếu quá dài
            words = prompt.split()
            lines = []
            current_line = ""
            for word in words:
                if len(current_line) + len(word) > 40:
                    lines.append(current_line)
                    current_line = word
                else:
                    current_line += " " + word if current_line else word
            if current_line:
                lines.append(current_line)
            
            # Vẽ text
            y_offset = 150
            for line in lines:
                cv2.putText(img, line, (50, y_offset), font, font_scale, color, thickness)
                y_offset += 60
            
            # Lưu ảnh
            img_path = f'{output_dir}/frame_{idx:03d}.png'
            cv2.imwrite(img_path, img)
            image_paths.append(img_path)
            logger.info(f"✅ Tạo frame {idx + 1}: {img_path}")
        
        return image_paths
    
    except Exception as e:
        logger.error(f"❌ Lỗi tạo ảnh: {e}")
        return []

print("✅ Hàm create_placeholder_images đã được định nghĩa")

### 4.6: Thêm Subtitle

In [ ]:
def add_subtitles_to_video(video_path: str, subtitle_text: str, output_path: str = None) -> str:
    """
    Thêm phụ đề vào video
    """
    try:
        if not output_path:
            output_path = video_path.replace('.mp4', '_subtitled.mp4')
        
        logger.info(f"📝 Thêm phụ đề vào video...")
        
        # Load video
        video = VideoFileClip(video_path)
        
        # Tạo text clip cho phụ đề
        txt_clip = TextClip(subtitle_text, fontsize=24, color='white', font='Arial')
        txt_clip = txt_clip.set_position('bottom').set_duration(video.duration)
        
        # Composite video
        result = CompositeVideoClip([video, txt_clip])
        result.write_videofile(output_path, verbose=False, logger=None)
        
        logger.info(f"✅ Video với phụ đề: {output_path}")
        
        # Giải phóng bộ nhớ
        video.close()
        result.close()
        
        return output_path
    
    except Exception as e:
        logger.error(f"❌ Lỗi thêm phụ đề: {e}")
        return video_path

print("✅ Hàm add_subtitles_to_video đã được định nghĩa")

### 4.7: Hàm chính - Xử lý từng URL

In [ ]:
def process_url(url: str, language: str = 'vi', voice: str = None, output_dir: str = None) -> Dict:
    """
    Xử lý URL từ A-Z: scrape -> summarize -> tts -> create video -> add subtitles
    """
    try:
        if not output_dir:
            output_dir = '/content/drive/MyDrive/colab-video-generator/videos'
        
        os.makedirs(output_dir, exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        result = {
            'url': url,
            'status': 'processing',
            'timestamp': timestamp,
            'steps': {}
        }
        
        # BƯỚC 1: Web Scraping
        logger.info(f"\n{'='*60}")
        logger.info(f"🔄 BƯỚC 1: WEB SCRAPING")
        logger.info(f"{'='*60}")
        scraped = scrape_content(url)
        if not scraped:
            result['status'] = 'failed'
            result['error'] = 'Scraping failed'
            return result
        result['steps']['scraping'] = scraped
        content = scraped['content']
        
        # BƯỚC 2: Tóm tắt với Gemini
        logger.info(f"\n{'='*60}")
        logger.info(f"🔄 BƯỚC 2: SUMMARIZATION (GEMINI)")
        logger.info(f"{'='*60}")
        summary_data = summarize_with_gemini(content, language)
        result['steps']['summary'] = summary_data
        
        scenes = summary_data.get('scenes', [])
        if not scenes:
            logger.warning("⚠️ Không có scene nào được tạo, tạo scene mặc định...")
            scenes = [{
                'title': 'Introduction',
                'script': content[:500],
                'visual_prompt': 'Professional introduction scene',
                'duration': 10
            }]
        
        # BƯỚC 3: Tạo Video và Voice-over cho từng scene
        logger.info(f"\n{'='*60}")
        logger.info(f"🔄 BƯỚC 3: VIDEO & VOICEOVER GENERATION")
        logger.info(f"{'='*60}")
        
        video_clips = []
        
        for scene_idx, scene in enumerate(scenes, 1):
            logger.info(f"\n📹 Xử lý Scene {scene_idx}/{len(scenes)}: {scene.get('title', 'Untitled')}")
            
            # Tạo voice-over
            script = scene.get('script', '')
            voiceover_path = generate_voiceover_sync(
                script,
                language=language,
                voice=voice,
                output_path=f'{output_dir}/voiceover_scene_{scene_idx}.mp3'
            )
            
            if not voiceover_path:
                logger.error(f"❌ Không thể tạo voice-over cho scene {scene_idx}")
                continue
            
            # Tạo ảnh placeholder
            visual_prompts = [scene.get('visual_prompt', 'Scene video')]
            image_paths = create_placeholder_images(
                visual_prompts,
                num_images=3,
                output_dir=f'{output_dir}/frames_scene_{scene_idx}'
            )
            
            if not image_paths:
                logger.error(f"❌ Không thể tạo ảnh cho scene {scene_idx}")
                continue
            
            # Tạo video từ ảnh + âm thanh
            scene_video_path = f'{output_dir}/scene_{scene_idx}.mp4'
            scene_video = create_video_from_images(
                image_paths,
                voiceover_path,
                output_path=scene_video_path
            )
            
            if scene_video:
                video_clips.append(scene_video)
        
        if not video_clips:
            result['status'] = 'failed'
            result['error'] = 'No video clips created'
            return result
        
        # BƯỚC 4: Ghép tất cả video lại
        logger.info(f"\n{'='*60}")
        logger.info(f"🔄 BƯỚC 4: VIDEO MERGING")
        logger.info(f"{'='*60}")
        logger.info(f"🎬 Ghép {len(video_clips)} video clip...")
        
        final_video_path = f'{output_dir}/final_video_{timestamp}.mp4'
        
        try:
            clips = [VideoFileClip(clip) for clip in video_clips]
            final_video = concatenate_videoclips(clips)
            final_video.write_videofile(final_video_path, verbose=False, logger=None)
            
            # Giải phóng bộ nhớ
            for clip in clips:
                clip.close()
            final_video.close()
            
            logger.info(f"✅ Video cuối cùng: {final_video_path}")
        except Exception as e:
            logger.error(f"❌ Lỗi ghép video: {e}")
            final_video_path = video_clips[0] if video_clips else None
        
        # BƯỚC 5: Thêm phụ đề
        logger.info(f"\n{'='*60}")
        logger.info(f"🔄 BƯỚC 5: ADD SUBTITLES")
        logger.info(f"{'='*60}")
        
        all_scripts = ' '.join([scene.get('script', '') for scene in scenes])
        final_video_with_subs = add_subtitles_to_video(
            final_video_path,
            all_scripts[:100] + "...",
            output_path=f'{output_dir}/final_video_with_subs_{timestamp}.mp4'
        )
        
        # BƯỚC 6: Lưu metadata
        logger.info(f"\n{'='*60}")
        logger.info(f"🔄 BƯỚC 6: SAVE METADATA")
        logger.info(f"{'='*60}")
        
        metadata = {
            'url': url,
            'language': language,
            'voice': voice,
            'title': scraped.get('title', 'Untitled'),
            'timestamp': timestamp,
            'num_scenes': len(scenes),
            'final_video': final_video_with_subs,
            'summary': summary_data.get('summary', ''),
            'scenes': scenes
        }
        
        metadata_path = f'{output_dir}/metadata_{timestamp}.json'
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)
        
        logger.info(f"✅ Metadata lưu tại: {metadata_path}")
        
        # Giải phóng GPU
        torch.cuda.empty_cache()
        
        result['status'] = 'completed'
        result['output_video'] = final_video_with_subs
        result['metadata'] = metadata_path
        result['steps']['final'] = metadata
        
        logger.info(f"\n{'='*60}")
        logger.info(f"✅ HOÀN THÀNH! Video: {final_video_with_subs}")
        logger.info(f"{'='*60}\n")
        
        return result
    
    except Exception as e:
        logger.error(f"❌ Lỗi xử lý URL: {e}")
        result['status'] = 'failed'
        result['error'] = str(e)
        return result

print("✅ Hàm process_url đã được định nghĩa")

## 🎯 BƯỚC 5: BATCH PROCESSING - XỬ LÝ DANH SÁCH URL

Xử lý nhiều URL cùng một lúc

In [ ]:
def process_batch_urls(urls: List[str], language: str = 'vi', voice: str = None, output_dir: str = None) -> List[Dict]:
    """
    Xử lý danh sách URL tuần tự với checkpoint recovery
    """
    if not output_dir:
        output_dir = '/content/drive/MyDrive/colab-video-generator/videos'
    
    os.makedirs(output_dir, exist_ok=True)
    
    checkpoint_file = f'{output_dir}/batch_checkpoint.json'
    processed_urls = set()
    results = []
    
    # Load checkpoint nếu có
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
            processed_urls = set(checkpoint.get('processed', []))
            logger.info(f"📋 Checkpoint tìm thấy: {len(processed_urls)} URL đã xử lý")
    
    # Lọc URL chưa xử lý
    urls_to_process = [url for url in urls if url not in processed_urls]
    logger.info(f"📊 Tổng: {len(urls)} URL, Cần xử lý: {len(urls_to_process)} URL")
    
    for idx, url in enumerate(urls_to_process, 1):
        logger.info(f"\n{'#'*70}")
        logger.info(f"# [{idx}/{len(urls_to_process)}] Đang xử lý: {url}")
        logger.info(f"{'#'*70}\n")
        
        result = process_url(url, language, voice, output_dir)
        results.append(result)
        
        # Cập nhật checkpoint
        processed_urls.add(url)
        checkpoint_data = {
            'total': len(urls),
            'processed': list(processed_urls),
            'completed': len([r for r in results if r['status'] == 'completed']),
            'failed': len([r for r in results if r['status'] == 'failed']),
            'last_update': datetime.now().isoformat()
        }
        
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)
        
        logger.info(f"💾 Checkpoint cập nhật")
    
    # Tóm tắt kết quả
    logger.info(f"\n{'='*70}")
    logger.info(f"📊 SUMMARY")
    logger.info(f"{'='*70}")
    logger.info(f"✅ Hoàn thành: {len([r for r in results if r['status'] == 'completed'])}")
    logger.info(f"❌ Thất bại: {len([r for r in results if r['status'] == 'failed'])}")
    logger.info(f"{'='*70}\n")
    
    return results

print("✅ Hàm process_batch_urls đã được định nghĩa")

## 🎨 BƯỚC 6: GRADIO INTERFACE - GIAO DIỆN WEB

Tạo giao diện dễ sử dụng với Gradio

In [ ]:
import gradio as gr

# Định nghĩa các voice disponible
VOICE_OPTIONS = {
    'Vietnamese (Nam)': 'vi-VN-NhanNeural',
    'Vietnamese (Nữ)': 'vi-VN-HoaiMyNeural',
    'English (US - Female)': 'en-US-AriaNeural',
    'English (US - Male)': 'en-US-GuyNeural',
    'English (UK - Female)': 'en-GB-SoniaNeural',
    'English (UK - Male)': 'en-GB-RyanNeural',
    'Chinese': 'zh-CN-XiaoxuanNeural',
    'Japanese': 'ja-JP-NanamiNeural',
    'Korean': 'ko-KR-SunHiNeural',
    'Spanish': 'es-ES-ElviraNeural',
    'French': 'fr-FR-DeniseNeural',
    'German': 'de-DE-AmalaNeural',
}

LANGUAGE_OPTIONS = {
    'Vietnamese': 'vi',
    'English': 'en',
    'Chinese': 'zh',
    'Japanese': 'ja',
    'Korean': 'ko',
    'Spanish': 'es',
    'French': 'fr',
    'German': 'de',
}

def create_video_interface(urls_text: str, language: str, voice: str, progress=gr.Progress()) -> str:
    """
    Hàm callback cho Gradio interface
    """
    try:
        # Parse URLs
        urls = [url.strip() for url in urls_text.split('\n') if url.strip()]
        if not urls:
            return "❌ Vui lòng nhập ít nhất một URL"
        
        # Lấy language code
        lang_code = LANGUAGE_OPTIONS.get(language, 'vi')
        voice_code = VOICE_OPTIONS.get(voice, 'vi-VN-NhanNeural')
        
        output_dir = '/content/drive/MyDrive/colab-video-generator/videos'
        
        # Xử lý batch
        results = process_batch_urls(urls, language=lang_code, voice=voice_code, output_dir=output_dir)
        
        # Format output
        output_text = f"""✅ XỬ LÝ HOÀN THÀNH!\n\n"""
        output_text += f"📊 Tóm tắt:\n"
        output_text += f"- Tổng URL: {len(urls)}\n"
        output_text += f"- Thành công: {len([r for r in results if r['status'] == 'completed'])}\n"
        output_text += f"- Thất bại: {len([r for r in results if r['status'] == 'failed'])}\n\n"
        
        output_text += f"📁 Kết quả lưu tại:\n{output_dir}\n\n"
        
        for result in results:
            if result['status'] == 'completed':
                output_text += f"✅ {result['url']}\n"
                output_text += f"   → {result.get('output_video', 'N/A')}\n\n"
            else:
                output_text += f"❌ {result['url']}\n"
                output_text += f"   Error: {result.get('error', 'Unknown error')}\n\n"
        
        return output_text
    
    except Exception as e:
        return f"❌ Lỗi: {str(e)}"

# Tạo Gradio interface
with gr.Blocks(title="🎬 Colab Video Generator") as demo:
    gr.Markdown("# 🎬 AI-Powered Content to Video Generator")
    gr.Markdown("### Chuyển đổi bài viết thành video chuyên nghiệp với AI")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### ⚙️ Cấu Hình")
            
            urls_input = gr.Textbox(
                label="📝 Danh Sách URL",
                placeholder="https://example.com/article\nhttps://another.com/article",
                lines=10,
                max_lines=20
            )
            
            language_select = gr.Dropdown(
                choices=list(LANGUAGE_OPTIONS.keys()),
                value="Vietnamese",
                label="🌐 Ngôn Ngữ"
            )
            
            voice_select = gr.Dropdown(
                choices=list(VOICE_OPTIONS.keys()),
                value="Vietnamese (Nam)",
                label="🎤 Giọng Đọc"
            )
            
            submit_btn = gr.Button(
                "🚀 Generate Videos",
                variant="primary",
                size="lg"
            )
        
        with gr.Column():
            gr.Markdown("### 📊 Kết Quả")
            output_text = gr.Textbox(
                label="Output",
                interactive=False,
                lines=30,
                max_lines=50
            )
    
    # Event handler
    submit_btn.click(
        fn=create_video_interface,
        inputs=[urls_input, language_select, voice_select],
        outputs=output_text
    )
    
    gr.Markdown("---")
    gr.Markdown("""
    ### 📚 Hướng Dẫn Sử Dụng
    
    1. **Nhập URL**: Dán danh sách các URL bài viết (mỗi link một dòng)
    2. **Chọn Ngôn Ngữ**: Lựa chọn ngôn ngữ tương ứng
    3. **Chọn Giọng Đọc**: Lựa chọn giọng đọc yêu thích
    4. **Generate Videos**: Bấm nút để bắt đầu xử lý
    5. **Chờ Kết Quả**: Video sẽ được lưu vào Google Drive
    
    ⏱️ **Thời gian dự kiến**: 7-18 phút/URL tùy độ phức tạp
    """)

print("✅ Gradio interface đã được tạo")

## 🌐 BƯỚC 7: KHỞI ĐỘNG GRADIO SERVER

Chạy ô này để mở giao diện web

In [ ]:
# Khởi động Gradio server
print("\n" + "="*70)
print("🚀 GRADIO SERVER ĐANG KHỞI ĐỘNG...")
print("="*70 + "\n")

demo.launch(
    share=True,
    debug=False,
    show_error=True
)

print("\n✅ Server đã tắt")

---

## 📝 HƯỚNG DẪN NÂNG CAO

### Option 1: Xử lý URL thủ công (không cần Gradio)

Nếu bạn muốn test một URL cụ thể:

In [ ]:
# Ví dụ: Xử lý một URL
test_url = "https://example.com/article"  # Thay bằng URL thực
test_language = "vi"  # Tiếng Việt
test_voice = "vi-VN-NhanNeural"  # Giọng Nam Việt

# Uncomment dòng dưới để chạy
# result = process_url(test_url, language=test_language, voice=test_voice)
# print(json.dumps(result, indent=2, ensure_ascii=False))

### Option 2: Xử lý danh sách URL từ file

In [ ]:
# Ví dụ: Đọc danh sách URL từ file txt
# urls_file = '/content/drive/MyDrive/urls.txt'
# if os.path.exists(urls_file):
#     with open(urls_file, 'r') as f:
#         urls = [line.strip() for line in f if line.strip()]
#     results = process_batch_urls(urls, language='vi')
# else:
#     print(f"❌ File không tìm thấy: {urls_file}")

### Option 3: Xem checkpoint & tiếp tục từ nơi bị gián đoạn

In [ ]:
# Xem checkpoint hiện tại
checkpoint_file = '/content/drive/MyDrive/colab-video-generator/videos/batch_checkpoint.json'
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        checkpoint = json.load(f)
    print("📋 Checkpoint hiện tại:")
    print(json.dumps(checkpoint, indent=2, ensure_ascii=False))
else:
    print("❌ Chưa có checkpoint")

---

## 🎯 TROUBLESHOOTING

### Lỗi: Out of Memory (OOM)

In [ ]:
# Xóa cache GPU
torch.cuda.empty_cache()
import gc
gc.collect()
print("✅ Cache đã được xóa")

### Lỗi: API Key Invalid

In [ ]:
# Test Gemini API
try:
    genai.configure(api_key=os.environ.get('GEMINI_API_KEY'))
    model = genai.GenerativeModel('gemini-pro')
    response = model.generate_content("Test: Say 'Hello'")
    print(f"✅ Gemini API hoạt động: {response.text}")
except Exception as e:
    print(f"❌ Lỗi Gemini API: {e}")
    print("📝 Lấy API key từ: https://makersuite.google.com/app/apikeys")

### Kiểm tra GPU

In [ ]:
# Kiểm tra GPU
!nvidia-smi

# Kiểm tra PyTorch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

---

## 📚 THAM KHẢO

- [Gemini API](https://ai.google.dev/)
- [Edge-TTS](https://github.com/rany2/edge-tts)
- [MoviePy](https://zulko.github.io/moviepy/)
- [OpenCV](https://opencv.org/)
- [Trafilatura](https://trafilatura.readthedocs.io/)

---

**Happy Video Creating! 🎬✨**